# Skate ATL: GVHMR video mocap test on Nia (evaluation only)

This notebook runs `squall01337/mixamo-llm-mocap` (GVHMR + Blender 5.1) on **one** roller-skate video and retargets the motion to Nia (`assets/skaters/nia/skater.glb`). The result is a test clip, `mocap_test_push`, a comparison with the hand-keyed `push` clip, metrics, and a `provenance.json`.

**License limits. Read before you run.**
- GVHMR code and checkpoint: educational, research and non-profit use only. No commercial use.
- SMPL-X body model: non-commercial scientific research only. You must register at https://smpl-x.is.tue.mpg.de/ yourself and accept the license.
- YOLOv8x checkpoint: AGPL-3.0.
- **Do not ship the output in the game.** It is an evaluation clip.

**Before you run**
1. Runtime → Change runtime type → **T4, L4 or A100 GPU**.
2. Put `SMPLX_NEUTRAL.npz` (from SMPL-X v1.1, NPZ+PKL) in your Google Drive at `MyDrive/skate-atl-mocap/SMPLX_NEUTRAL.npz`.
3. Choose the video in the config cell (cell 1). Use only a video with a clear reuse license: CC0, public domain, CC BY, CC BY-SA (Wikimedia Commons) or the Pexels license. No YouTube or TikTok. Full body, steady camera, side or front view, one skater, 5 to 15 s.
4. Runtime → Run all. The run takes approximately 20 to 40 minutes (most of it is the install and the 5.3 GB checkpoint download).

The outputs go to `MyDrive/skate-atl-mocap/out/<time>/`. Claude reads them from there.

In [ ]:
# 1. Config
DRIVE_DIR = '/content/drive/MyDrive/skate-atl-mocap'
ATL_REPO = 'https://github.com/William-Hill/atl-arcade'
ATL_BRANCH = 'claude/wizardly-mayer-binhnd'
MOCAP_REPO = 'https://github.com/squall01337/mixamo-llm-mocap'
MOCAP_COMMIT = 'da5e7cc48557f0272832429a8df3fbf2b3c1273d'   # 2026-09-26
GVHMR_REPO = 'https://github.com/zju3dv/GVHMR'
GVHMR_COMMIT = 'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'   # 2026-09-15
BLENDER_URL = 'https://download.blender.org/release/Blender5.1/blender-5.1.2-linux-x64.tar.xz'
CKPT_BASE = 'https://huggingface.co/camenduru/GVHMR/resolve/main'  # unofficial mirror of the GVHMR checkpoints
EXCLUDE_NEWER = '2026-11-14'   # project rule: dependency versions 7 days old or older

CLIP_NAME = 'mocap_test_push'  # or 'mocap_test_coast'
REF_CLIP = 'push'              # or 'coast'

# Video source. Option A: Wikimedia Commons file title (the license is read from the API).
COMMONS_FILE = ''              # for example 'File:Some roller skating video.webm'
# Option B: a Pexels video. Pexels license: free use, no attribution required (we record it anyway).
PEXELS_PAGE = ''               # the page URL, for example 'https://www.pexels.com/video/...-1234567/'
PEXELS_FILE_URL = ''           # the direct .mp4 URL (Pexels page -> Free download -> copy the link)
PEXELS_CREATOR = ''            # the name on the Pexels page
# The part of the video to use (seconds). 5 to 15 s, one skater pushing or gliding.
T0, T1 = 0.0, 10.0

In [ ]:
# 2. GPU check: GVHMR needs a CUDA GPU with approximately 8 GB.
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, 'No GPU. Runtime -> Change runtime type -> select a GPU, then run again.'
GPU_NAME = r.stdout.split(',')[0].strip()
mem_mib = int(r.stdout.split(',')[1].split()[0])
assert mem_mib >= 7500, f'{GPU_NAME} has {mem_mib} MiB; GVHMR needs approximately 8 GB.'

In [ ]:
# 3. Google Drive and the SMPL-X body model (license-gated; not downloaded by this notebook)
from google.colab import drive
import os, hashlib
drive.mount('/content/drive')
SMPLX = os.path.join(DRIVE_DIR, 'SMPLX_NEUTRAL.npz')
assert os.path.exists(SMPLX), (
    f'{SMPLX} not found. Register at https://smpl-x.is.tue.mpg.de/, accept the license, '
    'download SMPL-X v1.1 (NPZ+PKL), and put SMPLX_NEUTRAL.npz in MyDrive/skate-atl-mocap/.')
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
SMPLX_SHA = sha256(SMPLX)
print('SMPL-X found, sha256', SMPLX_SHA)

In [ ]:
# 4. Code: Skate ATL (scripts/mocap), the pipeline, GVHMR, Blender 5.1.2
%cd /content
!rm -rf atl-arcade mocap
!git clone -q --depth 1 -b {ATL_BRANCH} {ATL_REPO} atl-arcade
!git clone -q {MOCAP_REPO} mocap && git -C mocap checkout -q {MOCAP_COMMIT}
!git clone -q {GVHMR_REPO} mocap/tools/GVHMR && git -C mocap/tools/GVHMR checkout -q {GVHMR_COMMIT}
ATL_COMMIT = !git -C atl-arcade rev-parse HEAD
ATL_COMMIT = ATL_COMMIT[0]
print('atl-arcade', ATL_COMMIT)
!git -C mocap log -1 --format='mocap %H %ad' --date=short
!git -C mocap/tools/GVHMR log -1 --format='GVHMR %H %ad' --date=short
if not os.path.exists('/content/blender/blender'):
    !mkdir -p /content/blender && wget -q -O /content/b.tar.xz {BLENDER_URL} && tar xf /content/b.tar.xz -C /content/blender --strip-components=1 && rm /content/b.tar.xz
!apt-get -qq install -y libegl1 libgl1 libxi6 libxkbcommon0 libxrender1 libsm6 > /dev/null
BLENDER = '/content/blender/blender'
!{BLENDER} --version | head -1

In [ ]:
# 5. GVHMR environment: Python 3.10 venv with uv (CUDA 12.1 wheels), and MediaPipe in the Colab Python
!pip -q install uv
VENV = '/content/gvhmr-venv'
PY = VENV + '/bin/python'
if not os.path.exists(PY):
    !uv venv -q -p 3.10 {VENV}
!uv pip install -q -p {PY} --index-url https://download.pytorch.org/whl/cu121 torch==2.3.0+cu121 torchvision==0.18.0+cu121
!uv pip install -q -p {PY} --exclude-newer {EXCLUDE_NEWER} https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl
!uv pip install -q -p {PY} -r /content/atl-arcade/scripts/mocap/gvhmr-requirements-lock.txt
!uv pip install -q -p {PY} --no-deps -e /content/mocap/tools/GVHMR
!uv pip install -q -p {PY} --exclude-newer {EXCLUDE_NEWER} yacs
!uv pip install -q --system --exclude-newer {EXCLUDE_NEWER} mediapipe opencv-python
%cd /content/mocap/tools/GVHMR
!{PY} -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0)); from hmr4d.utils.smplx_utils import make_smplx; from hmr4d.model.gvhmr.gvhmr_pl_demo import DemoPL; from hmr4d.utils.preproc import Tracker; print('GVHMR import chain OK')"
%cd /content

In [ ]:
# 6. Checkpoints (approximately 5.3 GB) from the Hugging Face mirror, and the SMPL-X file
import shutil
CK = '/content/mocap/tools/GVHMR/inputs/checkpoints'
FILES = {'gvhmr/gvhmr_siga24_release.ckpt': 'gvhmr/gvhmr_siga24_release.ckpt',
         'yolo/yolov8x.pt': 'yolo/yolov8x.pt',
         'vitpose/vitpose-h-multi-coco.pth': 'vitpose/vitpose-h-multi-coco.pth',
         'hmr2/epoch%3D10-step%3D25000.ckpt': 'hmr2/epoch=10-step=25000.ckpt'}
WEIGHTS = {}
for remote, local in FILES.items():
    dst = os.path.join(CK, local)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        !curl -sSL --fail -o "{dst}" "{CKPT_BASE}/{remote}"
    WEIGHTS[local] = {'url': f'{CKPT_BASE}/{remote}', 'bytes': os.path.getsize(dst), 'sha256': sha256(dst)}
    print(local, WEIGHTS[local]['bytes'])
os.makedirs(os.path.join(CK, 'body_models/smplx'), exist_ok=True)
shutil.copy(SMPLX, os.path.join(CK, 'body_models/smplx/SMPLX_NEUTRAL.npz'))

In [ ]:
# 7. Video candidates on Wikimedia Commons (with license data). Skip this cell if you use Pexels.
import requests, re, html
UA = {'User-Agent': 'skate-atl-mocap-eval/0.1 (research evaluation notebook)'}
API = 'https://commons.wikimedia.org/w/api.php'
def strip(s): return html.unescape(re.sub('<[^>]+>', '', s or '')).strip()
def commons_info(titles):
    r = requests.get(API, headers=UA, params={'action': 'query', 'titles': '|'.join(titles), 'prop': 'imageinfo',
        'iiprop': 'url|size|mime|extmetadata|metadata', 'format': 'json'}).json()
    out = []
    for p in r.get('query', {}).get('pages', {}).values():
        ii = (p.get('imageinfo') or [{}])[0]
        em = ii.get('extmetadata', {})
        meta = {m['name']: m['value'] for m in (ii.get('metadata') or []) if isinstance(m, dict) and 'name' in m}
        out.append({'title': p.get('title'), 'url': ii.get('url'), 'page': ii.get('descriptionurl'),
                    'mime': ii.get('mime'), 'w': ii.get('width'), 'h': ii.get('height'),
                    'seconds': meta.get('playtime_seconds') or meta.get('length'),
                    'license': strip(em.get('LicenseShortName', {}).get('value')),
                    'license_url': strip(em.get('LicenseUrl', {}).get('value')),
                    'creator': strip(em.get('Artist', {}).get('value')),
                    'credit': strip(em.get('Credit', {}).get('value'))})
    return out
titles = set()
for q in ['roller skating filetype:video', 'roller skate filetype:video', 'roller derby filetype:video', 'rink skating filetype:video']:
    r = requests.get(API, headers=UA, params={'action': 'query', 'list': 'search', 'srsearch': q,
                     'srnamespace': 6, 'srlimit': 30, 'format': 'json'}).json()
    titles |= {s['title'] for s in r.get('query', {}).get('search', [])}
r = requests.get(API, headers=UA, params={'action': 'query', 'list': 'categorymembers', 'cmtitle': 'Category:Videos of roller skating',
                 'cmtype': 'file', 'cmlimit': 50, 'format': 'json'}).json()
titles |= {m['title'] for m in r.get('query', {}).get('categorymembers', [])}
titles = sorted(titles)
CANDIDATES = []
for i in range(0, len(titles), 40):
    CANDIDATES += commons_info(titles[i:i + 40])
for c in CANDIDATES:
    print(f"{c['title']}\n    {c['license']} | {c['creator'][:60]} | {c['w']}x{c['h']} | {c['seconds']} s | {c['page']}")
print(len(CANDIDATES), 'candidates. Open the pages, choose one with a steady camera and a full-body skater, set COMMONS_FILE, T0, T1 in cell 1, run cell 1, then run from cell 8.')

In [ ]:
# 8. Download the chosen video, check its license, cut T0..T1, 30 fps, at most 1280 px wide
import json, datetime
ALLOWED = re.compile(r'^(CC0|Public domain|PD|CC BY(-SA)? [0-9.]+|CC-BY(-SA)?-[0-9.]+|Pexels License)', re.I)
PLATE_DIR = '/content/mocap/plates/skate'
os.makedirs(PLATE_DIR, exist_ok=True)
if COMMONS_FILE:
    info = commons_info([COMMONS_FILE])[0]
    assert info['url'], f'{COMMONS_FILE} not found on Commons'
    SOURCE = {'site': 'Wikimedia Commons', 'title': info['title'], 'page': info['page'], 'file_url': info['url'],
              'creator': info['creator'], 'credit': info['credit'], 'license': info['license'], 'license_url': info['license_url']}
elif PEXELS_FILE_URL:
    assert PEXELS_PAGE and PEXELS_CREATOR, 'set PEXELS_PAGE and PEXELS_CREATOR too'
    SOURCE = {'site': 'Pexels', 'title': PEXELS_PAGE.rstrip('/').rsplit('/', 1)[-1], 'page': PEXELS_PAGE,
              'file_url': PEXELS_FILE_URL, 'creator': PEXELS_CREATOR, 'license': 'Pexels License',
              'license_url': 'https://www.pexels.com/license/'}
else:
    raise SystemExit('Set COMMONS_FILE or the PEXELS_* values in cell 1, run cell 1, then run from this cell.')
assert ALLOWED.match(SOURCE['license']), f"license {SOURCE['license']!r} is not on the allowed list; choose another video"
assert 5.0 <= T1 - T0 <= 15.0, 'use 5 to 15 s of video'
SOURCE.update({'trim_s': [T0, T1], 'downloaded': datetime.datetime.utcnow().isoformat() + 'Z'})
src_file = os.path.join(PLATE_DIR, 'source' + os.path.splitext(SOURCE['file_url'].split('?')[0])[1])
FILE_URL = SOURCE['file_url']
!curl -sSL --fail -A "skate-atl-mocap-eval/0.1" -o "{src_file}" "{FILE_URL}"
SOURCE['sha256'] = sha256(src_file)
PLATE = os.path.join(PLATE_DIR, 'skate.mp4')
!ffmpeg -loglevel error -y -ss {T0} -to {T1} -i "{src_file}" -vf "scale='min(1280,iw)':-2,fps=30" -an -c:v libx264 -crf 18 -pix_fmt yuv420p "{PLATE}"
json.dump(SOURCE, open(os.path.join(PLATE_DIR, 'video_source.json'), 'w'), indent=1)
print(json.dumps(SOURCE, indent=1))
# Look at 4 frames: one skater, full body, steady camera.
import cv2, matplotlib.pyplot as plt
cap = cv2.VideoCapture(PLATE); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for k, f in enumerate([0, n // 3, 2 * n // 3, n - 1]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, img = cap.read()
    ax[k].imshow(img[..., ::-1]); ax[k].set_title(f'frame {f}'); ax[k].axis('off')
plt.show()

In [ ]:
# 9. GVHMR estimate (GPU). Writes plates/skate/smplx.npz (refined on the 2D keypoints, heels and toes from MediaPipe).
%cd /content/mocap
!GVHMR_ROOT=/content/mocap/tools/GVHMR {PY} pipeline/estimate_pose_gvhmr.py --video plates/skate/skate.mp4 --out plates/skate/landmarks.json --smplx-only --feet-python /usr/bin/python3
assert os.path.exists('plates/skate/smplx.npz'), 'GVHMR did not write smplx.npz; read the log above'

In [ ]:
# 10. Retarget to Nia. Support foot "none": every foot lock is off, because a lock stops the skate.
#     The rig profile gives the retarget Nia's rest pose in the Y-up armature space it expects.
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
!{BLENDER} -b --factory-startup -P /content/atl-arcade/scripts/mocap/nia_rig_profile.py -- --in /content/atl-arcade/assets/skaters/nia/skater.glb --out {WORK}/nia.json --save-blend {WORK}/nia_rest.blend 2>&1 | grep rig-profile
SPEC = {
    'name': 'skate', 'action_name': CLIP_NAME, 'clip_dir': 'clips/skate',
    'landmarks': 'plates/skate/landmarks.json', 'smplx': 'plates/skate/smplx.npz',
    'rig_profile': f'{WORK}/nia.json', 'estimator': 'gvhmr_siga24', 'src_fps': 30, 'dst_fps': 30,
    'smplx_retarget': {
        # support foot "none" for the whole clip:
        'pin_static_feet': False,      # no body correction from static feet
        'contact_threshold': 1.01,     # GVHMR static flags never count
        'contact_max_speed': 0.0,      # no "slow foot" contacts
        'contact_still_px': 0.0,       # no "still on screen" contacts
        'contact_swing_height': 10.0,  # no single-support contacts
        'contact_geo_speed': 0.0,      # no floor-geometry contacts
        'floor_snap': True,            # height only: a free foot hovering near the floor goes down to it
    },
}
json.dump(SPEC, open(f'{WORK}/skate_spec.json', 'w'), indent=1)
!python3 pipeline/retarget_smplx.py --spec {WORK}/skate_spec.json --rig-profile {WORK}/nia.json

In [ ]:
# 11. In place, wheel lock with leg IK, loop, bake, export GLB (scripts/mocap/nia_mocap_clip.py)
OUT = f'{WORK}/mocap-test'
os.makedirs(OUT, exist_ok=True)
!{BLENDER} -b --factory-startup -P /content/atl-arcade/scripts/mocap/nia_mocap_clip.py -- --in /content/atl-arcade/assets/skaters/nia/skater.glb --motion /content/mocap/clips/skate/motion.npz --name {CLIP_NAME} --out {OUT}/{CLIP_NAME}.glb --report {OUT}/clip_report.json 2>&1 | grep -vE "Draco|INFO|WARNING"
REPORT = json.load(open(f'{OUT}/clip_report.json'))
assert REPORT['glb_bytes'] < 10_000_000, 'GLB over 10 MB'

In [ ]:
# 12. Comparison with the hand-keyed clip: renders at 3 points of each loop (side and front) and metrics
speed = REPORT['source_speed_m_s'] if REPORT['source_speed_m_s'] > 0.3 else 3.0
!{BLENDER} -b --factory-startup -P /content/atl-arcade/scripts/mocap/nia_mocap_compare.py -- --ref /content/atl-arcade/assets/skaters/nia/skater.glb --ref-clip {REF_CLIP} --mocap {OUT}/{CLIP_NAME}.glb --mocap-clip {CLIP_NAME} --out {OUT}/compare --speed {speed} --fps 30 2>&1 | grep -E "compare\]|Error|Traceback"
from IPython.display import Image, display
display(Image(f'{OUT}/compare/compare_sheet.png'))
print(open(f'{OUT}/compare/metrics.json').read())

In [ ]:
# 13. Optional: the pipeline's own overlay of the retarget on the video (front view). It shows the raw
#     retarget (with travel), before the in-place step. A failure here does not stop the test.
try:
    !python3 pipeline/render_review.py --spec {WORK}/skate_spec.json --clip clips/skate --scene {WORK}/nia_rest.blend --blender {BLENDER} --views front --no-details 2>&1 | tail -5
except Exception as e:
    print('overlay failed:', e)
!ls -la clips/skate

In [ ]:
# 14. Provenance, then copy the outputs to Google Drive
import glob, time
bl_ver = !{BLENDER} --version
prov = {
    'asset': f'{CLIP_NAME}: GVHMR video mocap test clip on Nia (evaluation only, not shipped)',
    'date': datetime.date.today().isoformat(),
    'status': 'evaluation only. Do not ship: GVHMR and SMPL-X are licensed for non-commercial research only.',
    'tools': {
        'mixamo-llm-mocap': {'repo': MOCAP_REPO, 'commit': MOCAP_COMMIT, 'license': 'MIT'},
        'GVHMR': {'repo': GVHMR_REPO, 'commit': GVHMR_COMMIT,
                  'license': 'educational, research and non-profit use only; modifications open source; no commercial use'},
        'blender': bl_ver[0], 'atl-arcade': {'branch': ATL_BRANCH, 'commit': ATL_COMMIT,
            'scripts': ['scripts/mocap/nia_rig_profile.py', 'scripts/mocap/nia_mocap_clip.py', 'scripts/mocap/nia_mocap_compare.py']},
        'gpu': GPU_NAME,
    },
    'model_weights': {
        **{k: {**v, 'source': 'unofficial Hugging Face mirror (camenduru/GVHMR) of the GVHMR release checkpoints'} for k, v in WEIGHTS.items()},
        'SMPLX_NEUTRAL.npz': {'version': 'SMPL-X v1.1', 'sha256': SMPLX_SHA,
                              'license': 'MPI SMPL-X license: non-commercial scientific research only; not redistributed'},
    },
    'weight_licenses': {'gvhmr_siga24_release.ckpt': 'GVHMR license (non-commercial)', 'yolov8x.pt': 'AGPL-3.0 (Ultralytics)',
                        'vitpose-h-multi-coco.pth': 'Apache-2.0 code (ViTPose); trained on COCO',
                        'hmr2 epoch=10-step=25000.ckpt': 'MIT code (4D-Humans); outputs SMPL, so the SMPL license applies',
                        'mediapipe pose/hand landmarker': 'Apache-2.0'},
    'video_source': SOURCE,
    'retarget_spec': SPEC,
    'clip_report': REPORT,
    'outputs': {os.path.relpath(p, OUT): os.path.getsize(p) for p in glob.glob(f'{OUT}/**/*', recursive=True) if os.path.isfile(p)},
}
json.dump(prov, open(f'{OUT}/provenance.json', 'w'), indent=1)
shutil.copy(f'{PLATE_DIR}/video_source.json', OUT)
stamp = time.strftime('%Y%m%d-%H%M%S')
dst = os.path.join(DRIVE_DIR, 'out', stamp)
shutil.copytree(OUT, dst)
# Not for git, kept in your Drive only: the SMPL-X result, the raw retarget, and the overlay video.
os.makedirs(os.path.join(dst, 'drive-only'), exist_ok=True)
for p in ['plates/skate/smplx.npz', 'clips/skate/motion.npz', 'clips/skate/curves.json'] + glob.glob('clips/skate/review_*.mp4'):
    if os.path.exists(p):
        shutil.copy(p, os.path.join(dst, 'drive-only'))
print('outputs in', dst)
!ls -laR "{dst}"